# SDAHU: raw windows (1 and sparse 100), sampled train, streaming val/test

Этот ноутбук:
- загружает `train/val/test` из CSV с `MultiIndex(['fault_type', 'Datetime'])`
- корректно извлекает target из `DataFrame` с тем же `MultiIndex`
- строит **raw-window** признаки как все значения в окне, **внутри каждой траектории `fault_type`**
- для **train** генерирует окна только для **стратифицированного сэмпла**
- для **val/test** делает **streaming inference батчами**, без материализации всех окон в память
- сравнивает `Dummy`, `ExtraTrees`, `HistGradientBoosting` для окна `1`, `10`, `100` и `100`, где берутся только позиции `0, 10, ..., 90`.


In [1]:
import gc
import json
import os
import math
import time
import warnings
from os.path import join

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report


In [2]:

folder = r'/home/ilya_treyvish/projects/lbnl_fdd/data/processed/SDAHU/preprocessed_resampled_1'

train_df = pd.read_csv(join(folder, 'train_df.csv'), index_col=[0, 1], parse_dates=[1])
train_target = pd.read_csv(join(folder, 'train_target.csv'), index_col=[0, 1], parse_dates=[1])

val_df = pd.read_csv(join(folder, 'val_df.csv'), index_col=[0, 1], parse_dates=[1])
val_target = pd.read_csv(join(folder, 'val_target.csv'), index_col=[0, 1], parse_dates=[1])

test_df = pd.read_csv(join(folder, 'test_df.csv'), index_col=[0, 1], parse_dates=[1])
test_target = pd.read_csv(join(folder, 'test_target.csv'), index_col=[0, 1], parse_dates=[1])

print("train_df:", train_df.shape)
print("train_target:", train_target.shape)
print("val_df:", val_df.shape)
print("val_target:", val_target.shape)
print("test_df:", test_df.shape)
print("test_target:", test_target.shape)


train_df: (1482900, 25)
train_target: (1482900, 1)
val_df: (439200, 25)
val_target: (439200, 1)
test_df: (633121, 25)
test_target: (633121, 1)


In [3]:

train_df.head()


CHWC_VLV_DM  ZONE_TEMP_4    SA_CFM  \
fault_type Datetime                                                  
0          2018-01-01 01:00:00          0.0    66.761273 -0.943175   
           2018-01-01 01:03:00          0.0    66.760743 -0.942716   
           2018-01-01 01:06:00          0.0    66.759625 -0.942253   
           2018-01-01 01:09:00          0.0    66.757902 -0.941795   
           2018-01-01 01:12:00          0.0    66.755709 -0.941337   

                                      SF_WAT    MA_TEMP  SYS_CTL  OA_DMPR_DM  \
fault_type Datetime                                                            
0          2018-01-01 01:00:00 -2.790609e-13  66.374680      0.0         0.0   
           2018-01-01 01:03:00 -2.787898e-13  66.374626      0.0         0.0   
           2018-01-01 01:06:00 -2.785185e-13  66.374626      0.0         0.0   
           2018-01-01 01:09:00 -2.782476e-13  66.374626      0.0         0.0   
           2018-01-01 01:12:00 -2.779765e-13  66.374626      0.0         0.0   

                                ZONE_TEMP_2    RA_TEMP      CHWC_VLV  ...  \
fault_type Datetime                                                   ...   
0          2018-01-01 01:00:00    66.766641  68.347797  2.498414e-21  ...   
           2018-01-01 01:03:00    66.764512  68.364925  4.846456e-22  ...   
           2018-01-01 01:06:00    66.761822  68.380317 -4.066457e-23  ...   
           2018-01-01 01:09:00    66.758562  68.394087 -1.525659e-22  ...   
           2018-01-01 01:12:00    66.754811  68.406397 -6.144650e-23  ...   

                                RA_DMPR_DM    SA_TEMP  RF_CS  RA_DMPR  \
fault_type Datetime                                                     
0          2018-01-01 01:00:00         1.0  66.374680    0.0      1.0   
           2018-01-01 01:03:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:06:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:09:00         1.0  66.374626    0.0      1.0   
           2018-01-01 01:12:00         1.0  66.374626    0.0      1.0   

                                  RA_CFM  RF_SPD_DM        RF_WAT  \
fault_type Datetime                                                 
0          2018-01-01 01:00:00  0.853056        0.0 -2.616298e-13   
           2018-01-01 01:03:00  0.852641        0.0 -2.613705e-13   
           2018-01-01 01:06:00  0.852225        0.0 -2.611207e-13   
           2018-01-01 01:09:00  0.851811        0.0 -2.608611e-13   
           2018-01-01 01:12:00  0.851395        0.0 -2.606081e-13   

                                ZONE_TEMP_1  ZONE_TEMP_3  SF_SPD_DM  
fault_type Datetime                                                  
0          2018-01-01 01:00:00    73.977550    67.025858        0.0  
           2018-01-01 01:03:00    74.067213    67.021650        0.0  
           2018-01-01 01:06:00    74.151062    67.016646        0.0  
           2018-01-01 01:09:00    74.229578    67.010939        0.0  
           2018-01-01 01:12:00    74.303257    67.004563        0.0  

[5 rows x 25 columns]

In [4]:

train_target.head()


target
fault_type Datetime                   
0          2018-01-01 01:00:00       0
           2018-01-01 01:03:00       0
           2018-01-01 01:06:00       0
           2018-01-01 01:09:00       0
           2018-01-01 01:12:00       0

In [5]:

def ensure_target_series(target: pd.DataFrame | pd.Series | np.ndarray) -> pd.Series:
    """
    Приводит target к Series.
    Поддерживает:
    - Series
    - DataFrame с 1 колонкой
    - ndarray shape (n,) или (n, 1)
    """
    if isinstance(target, pd.Series):
        return target

    if isinstance(target, pd.DataFrame):
        if target.shape[1] != 1:
            raise ValueError(
                f"У target DataFrame {target.shape[1]} столбцов; ожидался 1 столбец."
            )
        return target.iloc[:, 0]

    arr = np.asarray(target)
    if arr.ndim == 2:
        if arr.shape[1] != 1:
            raise ValueError(f"target имеет shape={arr.shape}; ожидался shape (n,) или (n, 1).")
        arr = arr.reshape(-1)
    elif arr.ndim != 1:
        raise ValueError(f"target имеет ndim={arr.ndim}; ожидался 1D или 2D c одним столбцом.")

    return pd.Series(arr)


def prepare_grouped_arrays(df: pd.DataFrame, target) -> tuple[list, list]:
    """
    Возвращает список групп (по fault_type), где каждая группа содержит:
    - X: np.ndarray shape (T, F)
    - y: np.ndarray shape (T,)
    - index: исходный MultiIndex

    Здесь target может быть DataFrame с тем же MultiIndex, что и df.
    """
    x = df.copy().sort_index(level=["fault_type", "Datetime"])
    y = ensure_target_series(target)

    if len(y) != len(x):
        raise ValueError(
            f"Длины df и target не совпадают: len(df)={len(x)}, len(target)={len(y)}"
        )

    # Если target имеет тот же MultiIndex — выравниваем по нему.
    # Иначе считаем, что target уже в том же построчном порядке, что и df.
    if isinstance(y.index, pd.MultiIndex) and y.index.equals(x.index):
        y_sorted = y.loc[x.index]
    else:
        y_sorted = pd.Series(np.asarray(y).reshape(-1), index=x.index)

    num_cols = [c for c in x.columns if pd.api.types.is_numeric_dtype(x[c])]
    feature_names = num_cols

    groups = []
    for fault_value, g in x.groupby(level="fault_type", sort=False):
        g_idx = g.index
        Xg = g[num_cols].to_numpy(dtype=np.float32)
        yg = y_sorted.loc[g_idx].to_numpy()
        yg = np.asarray(yg).reshape(-1)

        groups.append({
            "fault_type": fault_value,
            "X": Xg,
            "y": yg,
            "index": g_idx,
        })

    return groups, feature_names


In [6]:

train_groups, feature_names = prepare_grouped_arrays(train_df, train_target)
val_groups, _ = prepare_grouped_arrays(val_df, val_target)
test_groups, _ = prepare_grouped_arrays(test_df, test_target)

train_target_s = ensure_target_series(train_target)
if isinstance(train_target_s.index, pd.MultiIndex):
    train_target_s = train_target_s.sort_index(level=["fault_type", "Datetime"])

classes_ = np.sort(pd.unique(train_target_s.to_numpy().reshape(-1)))
n_features = len(feature_names)

print("classes_:", classes_)
print("n_features:", n_features)
print("n_train_groups:", len(train_groups))
print("example X shape:", train_groups[0]["X"].shape)
print("example y shape:", train_groups[0]["y"].shape)
print("example y dtype:", train_groups[0]["y"].dtype)


classes_: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14]
n_features: 25
n_train_groups: 15
example X shape: (101740, 25)
example y shape: (101740,)
example y dtype: int64


In [7]:
def make_window_offsets(window: int, sparse_step: int | None = None) -> np.ndarray:
    """
    Возвращает смещения внутри окна, которые нужно взять.
    - sparse_step=None -> берём все точки окна
    - sparse_step=10, window=100 -> берём позиции 0, 10, ..., 90
    """
    if sparse_step is None:
        return np.arange(window, dtype=np.int64)

    offsets = np.arange(0, window, sparse_step, dtype=np.int64)
    if len(offsets) == 0:
        raise ValueError(
            f"Для window={window} и sparse_step={sparse_step} не получилось построить offsets"
        )
    return offsets


def valid_window_endpoints(group_length: int, window: int) -> np.ndarray:
    if group_length < window:
        return np.array([], dtype=np.int64)
    return np.arange(window - 1, group_length, dtype=np.int64)


def flatten_window(Xg: np.ndarray, end: int, window: int, offsets: np.ndarray) -> np.ndarray:
    start = end - window + 1
    return Xg[start + offsets].reshape(-1)


def build_sampled_train_windows(
    groups,
    window: int,
    total_samples: int,
    random_state: int = 42,
    sparse_step: int | None = None,
    save_selection_path: str | None = None,
):
    """
    Строит raw-window train выборку:
    - окно flatten в вектор длины len(offsets) * n_features
    - сэмпл стратифицирован по классам
    - генерируются только выбранные окна
    - при необходимости сохраняется точный список использованных окон
    """
    rng = np.random.default_rng(random_state)
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)

    class_to_candidates = {}
    for group_idx, g in enumerate(groups):
        endpoints = valid_window_endpoints(len(g["X"]), window)
        if len(endpoints) == 0:
            continue

        labels = np.asarray(g["y"]).reshape(-1)
        for end in endpoints:
            cls = labels[end]
            class_to_candidates.setdefault(cls, []).append((group_idx, int(end)))

    classes = sorted(class_to_candidates.keys())
    if not classes:
        raise ValueError(f"Не найдено ни одного окна для window={window}")

    base_quota = total_samples // len(classes)
    remainder = total_samples % len(classes)

    selected = []
    shortages = 0

    for i, cls in enumerate(classes):
        candidates = class_to_candidates[cls]
        need = base_quota + (1 if i < remainder else 0)
        take = min(need, len(candidates))
        if take > 0:
            idx = rng.choice(len(candidates), size=take, replace=False)
            selected.extend([candidates[j] for j in idx])
        shortages += max(0, need - take)

    if shortages > 0:
        remaining = []
        selected_set = set(selected)
        for cls in classes:
            for item in class_to_candidates[cls]:
                if item not in selected_set:
                    remaining.append(item)

        if remaining:
            take = min(shortages, len(remaining))
            idx = rng.choice(len(remaining), size=take, replace=False)
            selected.extend([remaining[j] for j in idx])

    rng.shuffle(selected)

    n_samples = len(selected)
    if n_samples == 0:
        raise ValueError(f"После сэмплирования не осталось окон для window={window}")

    flat_dim = len(offsets) * groups[0]["X"].shape[1]
    X = np.empty((n_samples, flat_dim), dtype=np.float32)
    y = np.empty(n_samples, dtype=np.int64)
    records = []

    for i, (group_idx, end) in enumerate(selected):
        g = groups[group_idx]
        X[i] = flatten_window(g["X"], end=end, window=window, offsets=offsets)
        y[i] = np.asarray(g["y"][end]).item()

        records.append({
            "sample_id": i,
            "group_idx": group_idx,
            "fault_type": g["fault_type"],
            "window": window,
            "sparse_step": -1 if sparse_step is None else sparse_step,
            "endpoint_pos": int(end),
            "start_pos": int(end - window + 1),
            "label": int(y[i]),
            "endpoint_time": str(g["index"][end][1]),
            "offsets": ','.join(map(str, offsets.tolist())),
        })

    selection_df = pd.DataFrame(records)

    if save_selection_path is not None:
        os.makedirs(os.path.dirname(save_selection_path), exist_ok=True)
        selection_df.to_json(save_selection_path, orient="records", lines=True)
        print(f"Saved train window selection to: {save_selection_path}")

    return X, y, selection_df, offsets


def stream_windows(groups, window: int, batch_size: int = 4096, sparse_step: int | None = None):
    """
    Генератор батчей окон для train/val/test.
    Ничего не материализует целиком.
    Yield: X_batch, y_batch
    """
    offsets = make_window_offsets(window=window, sparse_step=sparse_step)
    X_parts = []
    y_parts = []

    for g in groups:
        Xg = g["X"]
        yg = np.asarray(g["y"]).reshape(-1)

        if len(Xg) < window:
            continue

        for end in range(window - 1, len(Xg)):
            X_parts.append(flatten_window(Xg, end=end, window=window, offsets=offsets))
            y_parts.append(np.asarray(yg[end]).item())

            if len(X_parts) >= batch_size:
                yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)
                X_parts = []
                y_parts = []

    if X_parts:
        yield np.asarray(X_parts, dtype=np.float32), np.asarray(y_parts, dtype=np.int64)


def evaluate_streaming(
    model,
    groups,
    window: int,
    classes_: np.ndarray,
    batch_size: int = 4096,
    sparse_step: int | None = None,
):
    """
    Streaming evaluation для train/val/test.
    Поддерживает accuracy, macro_f1, weighted_f1, logloss.
    """
    y_true_all = []
    y_pred_all = []
    y_proba_all = []

    has_proba = hasattr(model, "predict_proba")

    for Xb, yb in stream_windows(groups, window=window, batch_size=batch_size, sparse_step=sparse_step):
        pred = model.predict(Xb)
        y_true_all.append(yb)
        y_pred_all.append(pred)

        if has_proba:
            proba = model.predict_proba(Xb)
            y_proba_all.append(proba)

    y_true = np.concatenate(y_true_all)
    y_pred = np.concatenate(y_pred_all)

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
    }

    if has_proba and len(y_proba_all) > 0:
        y_proba = np.concatenate(y_proba_all)
        metrics["logloss"] = log_loss(y_true, y_proba, labels=classes_)
    else:
        metrics["logloss"] = np.nan

    return metrics, y_true, y_pred


In [8]:
RANDOM_STATE = 42
TRAIN_SAMPLE_SIZE = 50_000
VAL_BATCH_SIZE = 4_096
TEST_BATCH_SIZE = 4_096
ARTIFACTS_DIR = join("/home/ilya_treyvish/projects/lbnl_fdd/data", "artifacts_raw_windows")


In [10]:
window=100
train_sample_size=TRAIN_SAMPLE_SIZE
random_state=RANDOM_STATE
val_batch_size=VAL_BATCH_SIZE
test_batch_size=TEST_BATCH_SIZE
sparse_step=None

In [11]:
X_train, y_train, train_window_selection, offsets = build_sampled_train_windows(
        train_groups,
        window=window,
        total_samples=train_sample_size,
        random_state=random_state,
        sparse_step=sparse_step,
        save_selection_path=None,
    )

In [12]:
X_train.shape

(50000, 2500)

In [15]:
X_train.nbytes / 1024**2

476.837158203125

In [16]:
X_train.dtype

dtype('float32')

In [9]:
def run_experiment_for_window(
    window: int,
    train_groups,
    val_groups,
    test_groups,
    train_sample_size: int,
    random_state: int,
    val_batch_size: int = 4096,
    test_batch_size: int = 4096,
    sparse_step: int | None = None,
    version_name: str | None = None,
    artifacts_dir: str | None = None,
):
    if version_name is None:
        version_name = f"raw_window_{window}" if sparse_step is None else f"raw_window_{window}_step{sparse_step}"

    selection_path = None
    if artifacts_dir is not None:
        selection_path = join(artifacts_dir, f"{version_name}_train_window_selection.jsonl")

    X_train, y_train, train_window_selection, offsets = build_sampled_train_windows(
        train_groups,
        window=window,
        total_samples=train_sample_size,
        random_state=random_state,
        sparse_step=sparse_step,
        save_selection_path=selection_path,
    )

    models = [
        ("DummyClassifier", DummyClassifier(strategy="prior")),
        ("ExtraTreesClassifier", ExtraTreesClassifier(
            n_estimators=200,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=random_state,
        )),
        ("HistGradientBoostingClassifier", HistGradientBoostingClassifier(
            max_iter=200,
            learning_rate=0.1,
            random_state=random_state,
        )),
    ]

    rows = []
    fitted_models = {}

    for model_name, model in models:
        print(f"\n=== {version_name} | {model_name} ===")
        t0 = time.perf_counter()
        model.fit(X_train, y_train)
        fit_seconds = time.perf_counter() - t0

        train_pred = model.predict(X_train)
        train_metrics = {
            "accuracy": accuracy_score(y_train, train_pred),
            "macro_f1": f1_score(y_train, train_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_train, train_pred, average="weighted", zero_division=0),
        }
        if hasattr(model, "predict_proba"):
            train_proba = model.predict_proba(X_train)
            train_metrics["logloss"] = log_loss(y_train, train_proba, labels=classes_)
        else:
            train_metrics["logloss"] = np.nan

        val_metrics, _, _ = evaluate_streaming(
            model,
            val_groups,
            window=window,
            classes_=classes_,
            batch_size=val_batch_size,
            sparse_step=sparse_step,
        )
        test_metrics, _, _ = evaluate_streaming(
            model,
            test_groups,
            window=window,
            classes_=classes_,
            batch_size=test_batch_size,
            sparse_step=sparse_step,
        )

        rows.append({
            "version": version_name,
            "window": window,
            "sparse_step": -1 if sparse_step is None else sparse_step,
            "effective_window_points": len(offsets),
            "model_name": model_name,
            "fit_seconds": fit_seconds,
            "train_samples": len(X_train),
            "train_features": X_train.shape[1],
            "train_accuracy": train_metrics["accuracy"],
            "train_macro_f1": train_metrics["macro_f1"],
            "train_weighted_f1": train_metrics["weighted_f1"],
            "train_logloss": train_metrics["logloss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_macro_f1": val_metrics["macro_f1"],
            "val_weighted_f1": val_metrics["weighted_f1"],
            "val_logloss": val_metrics["logloss"],
            "test_accuracy": test_metrics["accuracy"],
            "test_macro_f1": test_metrics["macro_f1"],
            "test_weighted_f1": test_metrics["weighted_f1"],
            "test_logloss": test_metrics["logloss"],
        })
        fitted_models[model_name] = model

    results_df = pd.DataFrame(rows).sort_values(
        ["val_macro_f1", "val_accuracy"], ascending=False
    ).reset_index(drop=True)

    artifacts = {
        "version": version_name,
        "window": window,
        "sparse_step": sparse_step,
        "offsets": offsets,
        "X_train_shape": X_train.shape,
        "y_train_shape": y_train.shape,
        "train_window_selection": train_window_selection,
        "train_window_selection_path": selection_path,
        "results_df": results_df,
        "models": fitted_models,
    }

    del X_train, y_train
    gc.collect()

    return artifacts


In [10]:
%%time
artifacts_raw_w100 = run_experiment_for_window(
    window=100,
    train_groups=train_groups,
    val_groups=val_groups,
    test_groups=test_groups,
    train_sample_size=TRAIN_SAMPLE_SIZE,
    random_state=RANDOM_STATE,
    val_batch_size=VAL_BATCH_SIZE,
    test_batch_size=TEST_BATCH_SIZE,
    sparse_step=None,
    version_name="resampled_raw_window_100",
    artifacts_dir=ARTIFACTS_DIR,
)

artifacts_raw_w100["results_df"]


Saved train window selection to: /home/ilya_treyvish/projects/lbnl_fdd/data/artifacts_raw_windows/resampled_raw_window_100_train_window_selection.jsonl

=== resampled_raw_window_100 | DummyClassifier ===

=== resampled_raw_window_100 | ExtraTreesClassifier ===

=== resampled_raw_window_100 | HistGradientBoostingClassifier ===
CPU times: user 1h 27min 21s, sys: 2min 4s, total: 1h 29min 26s
Wall time: 22min 31s


,version,window,sparse_step,effective_window_points,model_name,fit_seconds,train_samples,train_features,train_accuracy,train_macro_f1,train_weighted_f1,train_logloss,val_accuracy,val_macro_f1,val_weighted_f1,val_logloss,test_accuracy,test_macro_f1,test_weighted_f1,test_logloss
0,resampled_raw_window_100,100,-1,100,HistGradientBoostingClassifier,768.451158,50000,2500,0.99284,0.992848,0.992847,0.027677,0.901555,0.901481,0.901481,0.265621,0.858791,0.865637,0.859430,0.356877
1,resampled_raw_window_100,100,-1,100,ExtraTreesClassifier,62.258853,50000,2500,0.99984,0.999840,0.999840,0.041558,0.891701,0.891054,0.891054,0.343612,0.849814,0.855958,0.849487,0.400036
2,resampled_raw_window_100,100,-1,100,DummyClassifier,0.003746,50000,2500,0.06668,0.008335,0.008337,2.708050,0.066667,0.008333,0.008333,2.708050,0.069757,0.008694,0.009097,2.708046
